In [1]:
import pandas as pd
import os
os.chdir('..')
os.getcwd()

'/Users/gabriele/Desktop/interpret_earth_embeddings'

In [2]:
csv_path = os.path.join('data', 'dw_locations_2026-02-13-1659_year-2024_50m_spherical_100k_random_stratified.csv')
df = pd.read_csv(csv_path)

# Subset for selected samples
df = df[(df['random_sample'] == 1) | (df['lc_stratified_sample'] == 1)]
df.reset_index(drop=True, inplace=True)

In [3]:
skipped_ids = set()
if os.path.exists(os.path.join('logs', f'tessera_skipped.txt')):
    with open(os.path.join('logs', f'tessera_skipped.txt'), 'r') as f:
        for fn in f:
            try:
                rid_str = fn.split('/')[-1].split("_", 1)[0]
                rid = int(rid_str)
            except ValueError:
                pass
            skipped_ids.add(rid)

In [19]:
partial_ids = set()
if os.path.exists(os.path.join('logs', f'partial_tessera.txt')):
    with open(os.path.join('logs', f'partial_tessera.txt'), 'r') as f:
        for fn in f:
            try:
                rid_str = fn.split("/")[-1].split(" ")[0].split('_', 1)[0]
                rid = int(rid_str)
            except ValueError:
                pass
            partial_ids.add(rid)

In [20]:
df_missing = df[df.id.isin(skipped_ids)]
df_partial = df[df.id.isin(partial_ids)]

In [21]:
import geopandas as gpd
from shapely import wkt

# Convert geometry column from WKT string → shapely object
df_partial["geometry"] = df_partial["geometry"].apply(wkt.loads)
df_missing["geometry"] = df_missing["geometry"].apply(wkt.loads)

# Convert to GeoDataFrame
gdf_partial = gpd.GeoDataFrame(df_partial, geometry="geometry", crs="EPSG:4326")
gdf_missing = gpd.GeoDataFrame(df_missing, geometry="geometry", crs="EPSG:4326")

# Now plot interactively
m = gdf_partial.explore(color="blue", name="Partial")
gdf_missing.explore(m=m, color="red", name="Missing")

m